# Senior regression experiment

This notebook solves the same deterministic regression problem as `junior_regression.ipynb`, but uses explicit functions, parameters, and a single reusable preprocessing path.

In [2]:
from __future__ import annotations

import random
from collections.abc import Sequence

DATA_PATH = "/data/hardcoded/junior_regression.csv"
MODEL_PATH = "/models/hardcoded/senior_model.bin"
DEFAULT_SEED = 2026
DEFAULT_SPLIT = 21
FEATURE_SCALE = 3.0

In [3]:
def make_dataset(seed: int, sample_count: int = 30) -> tuple[list[float], list[float]]:
    """Generate the deterministic dataset used by both experiment notebooks."""
    generator = random.Random(seed)
    features = [index / 10.0 for index in range(sample_count)]
    targets = [
        2.75 * value + 1.25 + generator.uniform(-0.15, 0.15) for value in features
    ]
    return features, targets


def split_dataset(
    features: Sequence[float],
    targets: Sequence[float],
    split_at: int,
) -> tuple[list[float], list[float], list[float], list[float]]:
    """Split features and targets into deterministic train and test partitions."""
    return (
        list(features[:split_at]),
        list(targets[:split_at]),
        list(features[split_at:]),
        list(targets[split_at:]),
    )


def scale_features(features: Sequence[float], scale: float) -> list[float]:
    """Apply the single preprocessing transform used by train and test."""
    return [value / scale for value in features]


def fit_linear_regression(
    features: Sequence[float], targets: Sequence[float]
) -> tuple[float, float]:
    """Fit a one-feature least-squares linear regression."""
    mean_x = sum(features) / len(features)
    mean_y = sum(targets) / len(targets)
    centered_x = [value - mean_x for value in features]
    centered_y = [value - mean_y for value in targets]
    coefficient = sum(a * b for a, b in zip(centered_x, centered_y)) / sum(
        value * value for value in centered_x
    )
    intercept = mean_y - coefficient * mean_x
    return intercept, coefficient


def predict(
    features: Sequence[float], intercept: float, coefficient: float
) -> list[float]:
    """Generate predictions from a fitted linear regression."""
    return [intercept + coefficient * value for value in features]


def mean_squared_error(actual: Sequence[float], predicted: Sequence[float]) -> float:
    """Compute the regression mean squared error."""
    return sum(
        (actual_value - predicted_value) ** 2
        for actual_value, predicted_value in zip(actual, predicted)
    ) / len(actual)


def run_experiment(seed: int = DEFAULT_SEED) -> float:
    """Train and evaluate the deterministic senior pipeline."""
    features, targets = make_dataset(seed)
    train_features, train_targets, test_features, test_targets = split_dataset(
        features, targets, DEFAULT_SPLIT
    )
    train_features = scale_features(train_features, FEATURE_SCALE)
    test_features = scale_features(test_features, FEATURE_SCALE)
    intercept, coefficient = fit_linear_regression(train_features, train_targets)
    predictions = predict(test_features, intercept, coefficient)
    return mean_squared_error(test_targets, predictions)

In [4]:
final_mse = run_experiment()
print(f"data={DATA_PATH}, model={MODEL_PATH}, seed={DEFAULT_SEED}")
print(f"final_mse={final_mse:.12f}")

data=/data/hardcoded/junior_regression.csv, model=/models/hardcoded/senior_model.bin, seed=2026
final_mse=0.011202345146
